# 02a - Baseline ML classifiers: Random Forest and LightGBM

Trains four models (RF and LightGBM on Elliptic and Ethereum) and reports
standard classification metrics. Checkpoints are saved under
`models/saved/`.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from xai_blockchain_framework.config import CONFIG, PATHS, set_seed
from xai_blockchain_framework.data import (
    load_elliptic, preprocess_elliptic,
    load_ethereum, preprocess_ethereum,
)
from xai_blockchain_framework.models import (
    evaluate_ml, save_ml_model, train_lightgbm, train_random_forest,
)
from xai_blockchain_framework.utils import save_csv

set_seed()


## Prepare data splits

In [ ]:
ell = load_elliptic()
X_ell, y_ell, ts_ell = preprocess_elliptic(ell, drop_unknown=True, normalize=True)

# Temporal split: time steps 1-34 for train, 35-39 for val, 40-49 for test.
train_mask_ell = ts_ell <= 34
val_mask_ell   = (ts_ell >= 35) & (ts_ell <= 39)
test_mask_ell  = ts_ell >= 40

eth = load_ethereum()
X_eth_full, y_eth, _ = preprocess_ethereum(eth)
X_eth_tr, X_eth_te, y_eth_tr, y_eth_te = train_test_split(
    X_eth_full, y_eth, test_size=0.2, random_state=CONFIG.random_seed, stratify=y_eth,
)
X_eth_tr, X_eth_val, y_eth_tr, y_eth_val = train_test_split(
    X_eth_tr, y_eth_tr, test_size=0.2, random_state=CONFIG.random_seed, stratify=y_eth_tr,
)


## Train Random Forest and LightGBM on both datasets

In [ ]:
results = []

for dataset, (X_tr, X_val, X_te, y_tr, y_val, y_te) in [
    ('Elliptic', (X_ell[train_mask_ell], X_ell[val_mask_ell], X_ell[test_mask_ell],
                  y_ell[train_mask_ell], y_ell[val_mask_ell], y_ell[test_mask_ell])),
    ('Ethereum', (X_eth_tr, X_eth_val, X_eth_te, y_eth_tr, y_eth_val, y_eth_te)),
]:
    for model_name, trainer in [
        ('RandomForest', lambda Xtr, ytr: train_random_forest(Xtr, ytr)),
        ('LightGBM',     lambda Xtr, ytr: train_lightgbm(Xtr, ytr)),
    ]:
        print(f'--- {model_name} on {dataset} ---')
        model = trainer(X_tr, y_tr)
        val_report = evaluate_ml(model, X_val, y_val)
        threshold = val_report.threshold
        test_report = evaluate_ml(model, X_te, y_te, threshold=threshold)
        for split, rep in [('Val', val_report), ('Test', test_report)]:
            row = {'Dataset': dataset, 'Model': model_name, 'Split': split}
            row.update(rep.as_dict())
            results.append(row)
        save_ml_model(
            model,
            PATHS.models_dir / f'{dataset.lower()}_{model_name.lower()}.joblib',
        )

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
save_csv(results_df, PATHS.results_dir / 'ml_baselines_results.csv')


## Save splits for downstream notebooks

In [ ]:
np.savez(
    PATHS.data_processed / 'elliptic_splits.npz',
    X_train=X_ell[train_mask_ell], X_val=X_ell[val_mask_ell], X_test=X_ell[test_mask_ell],
    y_train=y_ell[train_mask_ell], y_val=y_ell[val_mask_ell], y_test=y_ell[test_mask_ell],
    X_all=X_ell, y_all=y_ell, ts_all=ts_ell,
    train_mask=train_mask_ell, val_mask=val_mask_ell, test_mask=test_mask_ell,
)
np.savez(
    PATHS.data_processed / 'ethereum_splits.npz',
    X_train=X_eth_tr, X_val=X_eth_val, X_test=X_eth_te,
    y_train=y_eth_tr, y_val=y_eth_val, y_test=y_eth_te,
)
print('Splits saved to data/processed/')
